# 04 · Optimisers: momentum, RMSProp, Adam
NumPy for the model · matplotlib for graphs · Runs on Google Colab

1. The problem: plain steps struggle in a ravine
2. Momentum: a rolling ball
3. RMSProp: a separate step size for every weight
4. Adam = momentum + RMSProp
5. In our network code (digits)
6. Real use case: MNIST with a deeper network
7. Experiments

## Setup: one `optimizer_step` that works for any list of weights

In [ ]:
import io, time, urllib.request
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
np.set_printoptions(precision=4, suppress=True)
np.seterr(all="ignore")                                           # diverging runs overflow; we measure that instead

def new_state(params, beta=0.9):
    zeros = lambda: [np.zeros_like(p) for p in params]
    return {"t": 0, "beta": beta, "v": zeros(), "s": zeros(), "m": zeros()}

def optimizer_step(params, grads, state, name, lr):
    state["t"] += 1
    t, new = state["t"], []
    for k, (p, g) in enumerate(zip(params, grads)):
        if name == "sgd":                                         # plain step
            p = p - lr * g
        elif name == "momentum":                                  # velocity keeps the direction
            state["v"][k] = state["beta"] * state["v"][k] - lr * g
            p = p + state["v"][k]
        elif name == "rmsprop":                                   # divide by each weight's typical slope size
            state["s"][k] = 0.9 * state["s"][k] + 0.1 * g ** 2
            p = p - lr * g / (np.sqrt(state["s"][k]) + 1e-8)
        else:                                                     # adam: momentum + rmsprop + start-up correction
            state["m"][k] = 0.9 * state["m"][k] + 0.1 * g
            state["s"][k] = 0.999 * state["s"][k] + 0.001 * g ** 2
            m_hat = state["m"][k] / (1 - 0.9 ** t)
            s_hat = state["s"][k] / (1 - 0.999 ** t)
            p = p - lr * m_hat / (np.sqrt(s_hat) + 1e-8)
        new.append(p)
    return new

---
# Part 1: the problem — a ravine

$\text{loss} = \tfrac{1}{2}(w_1^2 + 100\,w_2^2)$: flat along $w_1$, 100× steeper along $w_2$. Slopes: $(w_1,\; 100\,w_2)$. Bottom at (0, 0).

In [ ]:
ravine_loss = lambda w: 0.5 * (w[0] ** 2 + 100 * w[1] ** 2)
ravine_grad = lambda w: np.array([w[0], 100 * w[1]])
START = np.array([-4.0, 1.0])

def ravine_path(name, lr, beta=0.9, steps=500):
    params = [START.copy()]
    state = new_state(params, beta)
    path = [START.copy()]
    for _ in range(steps):
        params = optimizer_step(params, [ravine_grad(params[0])], state, name, lr)
        path.append(params[0])
        if ravine_loss(params[0]) < 1e-3 or ravine_loss(params[0]) > 1e6:
            break
    return np.array(path)

def result(path):
    final = ravine_loss(path[-1])
    return f"reached the bottom in {len(path) - 1} steps" if final < 1e-3 else ("BLEW UP" if final > 1e6 else f"not there after {len(path) - 1} steps (loss {final:.3f})")

w1g, w2g = np.meshgrid(np.linspace(-4.5, 4.5, 200), np.linspace(-1.5, 1.5, 200))
LOSS_MAP = 0.5 * (w1g ** 2 + 100 * w2g ** 2)

def draw_paths(ax, paths, title):
    ax.contour(w1g, w2g, LOSS_MAP, levels=np.logspace(-2, 2.5, 20), cmap="Greys")
    for label, path in paths.items():
        ax.plot(path[:, 0], path[:, 1], "o-", ms=2, lw=1, label=label)
    ax.plot(0, 0, "g*", ms=14)
    ax.set(xlim=(-4.5, 4.5), ylim=(-1.5, 1.5), xlabel="w1 (flat)", ylabel="w2 (steep)", title=title)
    ax.legend(fontsize=8, loc="upper right")

In [ ]:
path = ravine_path("sgd", 0.019, steps=5)
print("Plain steps, η = 0.019: w2 flips sign every step (steep), w1 barely moves (flat)")
for k, w in enumerate(path):
    print(f"  step {k}: w1 = {w[0]:+.4f}, w2 = {w[1]:+.4f}")

fig, axes = plt.subplots(1, 3, figsize=(20, 4.5))
for ax, lr in zip(axes, [0.005, 0.019, 0.021]):
    p = ravine_path("sgd", lr)
    print(f"η = {lr}: {result(p)}")
    draw_paths(ax, {f"η = {lr}": p[:200]}, f"Plain steps, η = {lr}: {result(p)}")
plt.tight_layout()
plt.show()

Small η: crawls along the flat direction. η above 0.02: blows up in the steep direction. **No single η suits both directions.**

---
# Part 2: momentum — a rolling ball
$$v \leftarrow \beta\,v - \eta \cdot \text{slope} \qquad w \leftarrow w + v$$
Steps that keep pointing the same way add up; zig-zags cancel out.

## By hand: one weight, loss $= w^2$ (slope $2w$), η = 0.05, β = 0.9

In [ ]:
w_plain, w_mom, v = 4.0, 4.0, 0.0
print(f"{'step':>4} | {'plain w':>8} | {'slope':>7} | {'v = 0.9·v - 0.05·slope':>26} | {'momentum w':>10}")
print("-" * 68)
for k in range(1, 7):
    w_plain = w_plain - 0.05 * 2 * w_plain
    slope = 2 * w_mom
    new_v = 0.9 * v - 0.05 * slope
    print(f"{k:>4} | {w_plain:8.4f} | {slope:7.3f} | {f'0.9×{v:+.3f} - 0.05×{slope:.3f} = {new_v:+.3f}':>26} | {w_mom + new_v:10.4f}")
    v, w_mom = new_v, w_mom + new_v

Momentum builds up speed (v grows), so it moves faster than the plain step.

In [ ]:
paths = {}
for beta in [0.0, 0.5, 0.9, 0.95]:
    paths[f"β = {beta}"] = ravine_path("momentum", 0.015, beta)
    print(f"momentum, η = 0.015, β = {beta}: {result(paths[f'β = {beta}'])}")
fig, ax = plt.subplots(figsize=(10, 4))
draw_paths(ax, {k: p[:150] for k, p in paths.items()}, "Momentum on the ravine (η = 0.015)")
plt.show()

---
# Part 3: RMSProp — a separate step size for every weight
$$s \leftarrow 0.9\,s + 0.1\,\text{slope}^2 \qquad w \leftarrow w - \eta \cdot \frac{\text{slope}}{\sqrt{s}}$$
Steep weight ➜ big $s$ ➜ smaller steps. Flat weight ➜ small $s$ ➜ bigger steps.

In [ ]:
params = [START.copy()]
state = new_state(params)
for k in range(1, 4):
    g = ravine_grad(params[0])
    before = params[0].copy()
    params = optimizer_step(params, [g], state, "rmsprop", 0.2)
    step_taken = params[0] - before
    print(f"step {k}: slope = {g} | √s = {np.sqrt(state['s'][0])} | step taken = {step_taken} | step/slope = {step_taken / -g}")

p_rms = ravine_path("rmsprop", 0.2)
print(f"\nRMSProp, η = 0.2: {result(p_rms)}")

The steep $w_2$ gets a tiny step per unit of slope, the flat $w_1$ a big one, so both directions move at a similar pace.

---
# Part 4: Adam = momentum + RMSProp
$$m \leftarrow 0.9\,m + 0.1\,\text{slope} \qquad s \leftarrow 0.999\,s + 0.001\,\text{slope}^2 \qquad w \leftarrow w - \eta \cdot \frac{\hat{m}}{\sqrt{\hat{s}}}$$
$m$ and $s$ start at 0, so early on they are too small. The **start-up correction** fixes that: $\hat{m} = m / (1 - 0.9^t)$, $\hat{s} = s / (1 - 0.999^t)$.

In [ ]:
g = 2.0                                                           # suppose the slope is 2.0 every step
m = s = 0.0
for t in range(1, 4):
    m, s = 0.9 * m + 0.1 * g, 0.999 * s + 0.001 * g ** 2
    print(f"t = {t}: m = {m:.4f} (true slope 2.0) -> corrected m̂ = {m:.4f} / {1 - 0.9 ** t:.4f} = {m / (1 - 0.9 ** t):.4f} | "
          f"s = {s:.6f} -> ŝ = {s / (1 - 0.999 ** t):.4f} (true slope² 4.0)")

In [ ]:
all_paths = {"plain (best η 0.019)": ravine_path("sgd", 0.019), "momentum (η 0.015)": ravine_path("momentum", 0.015),
             "rmsprop (η 0.2)": ravine_path("rmsprop", 0.2), "adam (η 0.3)": ravine_path("adam", 0.3)}
for name, p in all_paths.items():
    print(f"{name:>22}: {result(p)}")
fig, ax = plt.subplots(figsize=(11, 4.5))
draw_paths(ax, {k: p[:120] for k, p in all_paths.items()}, "All four on the ravine (first 120 steps)")
plt.show()

---
# Part 5: in our network code
Same vectorised functions as the mini-batch notebook, except `backward_pass` now only **computes** the slopes and `optimizer_step` applies them.

In [ ]:
ACT = {
    "tanh": (np.tanh,                    lambda S: 1 - np.tanh(S) ** 2),
    "relu": (lambda S: np.maximum(0, S), lambda S: np.where(S > 0, 1.0, 0.0)),
}

def add_x0(X):
    return np.hstack([-np.ones((len(X), 1)), X])

def new_net(sizes, act="tanh"):
    k = 1 if act == "tanh" else 2                                 # Xavier for tanh, He for relu
    return {"act": act, "W": [rng.normal(0, np.sqrt(k / a), (a + 1, b)) for a, b in zip(sizes[:-1], sizes[1:])]}

def forward_pass(X0, net):
    f = ACT[net["act"]][0]
    H, cache = X0, []
    for W in net["W"][:-1]:
        S = H @ W
        cache.append((H, S))
        H = add_x0(f(S))
    cache.append((H, None))
    S = H @ net["W"][-1]
    E = np.exp(S - S.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True), cache

def compute_error(Y, P):
    return Y - P

def backward_pass(net, error, cache):
    df = ACT[net["act"]][1]
    delta, grads = error, [None] * len(net["W"])
    for l in range(len(net["W"]) - 1, -1, -1):
        grads[l] = -cache[l][0].T @ delta / len(delta)            # dW: average slope over the batch
        if l > 0:
            delta = (delta @ net["W"][l][1:].T) * df(cache[l - 1][1])
    return grads

def accuracy(net, X0, y):
    return 100 * (forward_pass(X0, net)[0].argmax(axis=1) == y).mean()

def train(X0, Y, net, name, lr, epochs, batch_size, X_test, y_test, beta=0.9):
    state, accs = new_state(net["W"], beta), []
    for _ in range(epochs):
        idx = rng.permutation(len(X0))
        for s in range(0, len(X0), batch_size):
            batch = idx[s:s + batch_size]
            P, cache = forward_pass(X0[batch], net)
            grads = backward_pass(net, compute_error(Y[batch], P), cache)
            net = {"act": net["act"], "W": optimizer_step(net["W"], grads, state, name, lr)}
        accs.append(accuracy(net, X_test, y_test))
    return net, accs

def load_digits(name):
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/optdigits/" + name
    d = np.genfromtxt(io.StringIO(urllib.request.urlopen(url).read().decode()), delimiter=",").astype(int)
    return d[:, :64] / 16, d[:, 64]

Xd_tr, yd_tr = load_digits("optdigits.tra")
Xd_te, yd_te = load_digits("optdigits.tes")
Xd_tr0, Xd_te0, Yd_tr = add_x0(Xd_tr), add_x0(Xd_te), np.eye(10)[yd_tr]
print(f"digits: train {Xd_tr.shape}, test {Xd_te.shape}")

## Digits `[64, 64, 64, 10]`, batch 32, each optimiser with its usual learning rate

In [ ]:
DEFAULT_LR = {"sgd": 0.1, "momentum": 0.01, "rmsprop": 0.001, "adam": 0.001}
fig, ax = plt.subplots(figsize=(8, 3.5))
for name, lr in DEFAULT_LR.items():
    _, accs = train(Xd_tr0, Yd_tr, new_net([64, 64, 64, 10]), name, lr, 10, 32, Xd_te0, yd_te)
    print(f"{name:>9} (η = {lr}): test accuracy per epoch {np.round(accs, 1)}")
    ax.plot(range(1, 11), accs, "o-", label=f"{name} (η = {lr})")
ax.set(xlabel="epoch", ylabel="test accuracy (%)", ylim=(85, 100), title="Digits: all four reach a similar accuracy")
ax.legend()
plt.show()

---
# Part 6: real use case — MNIST, `[784, 256, 128, 64, 10]` (ReLU, He), batch 64, 3 epochs

In [ ]:
mnist = np.load(io.BytesIO(urllib.request.urlopen("https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz").read()))
Xm_tr0, ym_tr = add_x0(mnist["x_train"].reshape(-1, 784) / 255), mnist["y_train"]
Xm_te0, ym_te = add_x0(mnist["x_test"].reshape(-1, 784) / 255), mnist["y_test"]
Ym_tr = np.eye(10)[ym_tr]

print(f"{'optimiser':>9} | {'η':>6} | test accuracy per epoch | seconds")
print("-" * 58)
for name, lr in DEFAULT_LR.items():
    start = time.time()
    _, accs = train(Xm_tr0, Ym_tr, new_net([784, 256, 128, 64, 10], "relu"), name, lr, 3, 64, Xm_te0, ym_te)
    print(f"{name:>9} | {lr:>6} | {str(np.round(accs, 2)):>23} | {time.time() - start:6.1f}")

With a well-chosen η, all four reach about 97% on MNIST. Adam does a little more work per step (two running averages), so each epoch is slower.

---
# Part 7: Experiments (digits, `[64, 64, 64, 10]`, batch 32, 5 epochs)
## 1. Learning-rate sensitivity

In [ ]:
names = list(DEFAULT_LR)
lrs = [1e-4, 1e-3, 1e-2, 1e-1, 1]
grid = np.array([[train(Xd_tr0, Yd_tr, new_net([64, 64, 64, 10]), n, lr, 5, 32, Xd_te0, yd_te)[1][-1] for lr in lrs] for n in names])

fig, ax = plt.subplots(figsize=(7, 4))
ax.imshow(grid, cmap="Greens", vmin=0, vmax=100)
for r in range(len(names)):
    for c in range(len(lrs)):
        ax.text(c, r, f"{grid[r, c]:.0f}%", ha="center", va="center")
ax.set(xticks=range(len(lrs)), xticklabels=lrs, yticks=range(len(names)), yticklabels=names,
       xlabel="learning rate η", title="Test accuracy after 5 epochs")
plt.show()

Each optimiser has its own good range of η: plain SGD and momentum want **large** η (0.01 and up), RMSProp and Adam want **small** η (0.0001–0.01). Adam's usual default 0.001 already sits in its good range, which is why it is a popular first choice.

## 2. Momentum β (η = 0.01)

In [ ]:
for beta in [0.0, 0.5, 0.9, 0.99]:
    _, accs = train(Xd_tr0, Yd_tr, new_net([64, 64, 64, 10]), "momentum", 0.01, 5, 32, Xd_te0, yd_te, beta)
    print(f"β = {beta:<4}: test accuracy per epoch {np.round(accs, 1)}")

## 3. Deep network: 6 hidden layers of 64, SGD vs Adam (3 random starts each)

In [ ]:
deep_sizes = [64] + [64] * 6 + [10]
for name, lr in [("sgd", 0.1), ("adam", 0.001)]:
    runs = [train(Xd_tr0, Yd_tr, new_net(deep_sizes), name, lr, 5, 32, Xd_te0, yd_te)[1][-1] for _ in range(3)]
    print(f"{name:>4} (η = {lr}): test accuracies {np.round(runs, 1)} | mean {np.mean(runs):.1f}%")

Next notebook: **overfitting and regularisation**.